# Self-Improvement & MLflow, Part 2

The code from [the article](https://opendatascience.com/self-improvement-mlflow-part-2/), in article order. Each code cell is also saved as a file in this folder.

The snippets show the MLflow calls, not a complete agent: names such as `agent` or `run_agent`, and the helpers marked *our helper*, are yours to supply. Each cell says which names it expects. A snippet that continues an earlier one doesn't repeat its imports.

## Version the prompt and load it by alias

File: `support_agent.py`

Not defined here: `agent` and `route_topic` are your own agent's code.

In [ ]:
import mlflow
from mlflow.entities import SpanType

mlflow.set_experiment("support-agent")
mlflow.openai.autolog()

v1 = mlflow.genai.register_prompt(
    name="support-system-prompt",
    template="You are a support agent for {{ product }}. Answer the customer's "
             "question using the policy tools available to you. "
             "Question: {{ question }}",
    commit_message="v1, initial prompt",
)

# The agent loads by alias, so promotion is a pointer move, not a code change.
mlflow.genai.set_prompt_alias(
    name="support-system-prompt", alias="production", version=v1.version
)


@mlflow.trace(span_type=SpanType.AGENT)
def run_agent(question: str) -> str:
    prompt = mlflow.genai.load_prompt(
        "prompts:/support-system-prompt@production"
    )
    mlflow.update_current_trace(tags={"topic": route_topic(question)})
    return agent.invoke(prompt.format(product="Acme", question=question))


## Judge last week's traces in place

File: `judge_recent_traces.py`

Not defined here: `leads_with_answer` comes from Part 1's `scorers.py`.

In [ ]:
import time

since = int((time.time() - 7 * 24 * 3600) * 1000)
recent = mlflow.search_traces(
    filter_string=f"attributes.timestamp_ms > {since}",
    return_type="list",
)

# Judge the week's traces in place. No predict_fn, so nothing is re-run.
mlflow.genai.evaluate(data=recent, scorers=[leads_with_answer])


## Grow the regression dataset

File: `regression_dataset.py`

Not defined here: `experiment_id` is your experiment's id.

In [ ]:
from mlflow.genai.datasets import create_dataset

dataset = create_dataset(
    name="support_regressions", experiment_id=[experiment_id]
)

failing = mlflow.search_traces(
    filter_string="feedback.quality = 'Fail' "
                  "AND expectation.expected_response IS NOT NULL",
    return_type="list",
)
dataset = dataset.merge_records(failing)

dataset = dataset.merge_records([
    {"inputs": {"question": "How long can I wait to request a refund?"},
     "expectations": {"expected_response": "Within 14 days of purchase."}},
    {"inputs": {"question": "I can't log in after resetting my password."},
     "expectations": {"expected_response": "Clear the browser cache and retry the "
                                           "reset link within 15 minutes."}},
])


## Register a candidate prompt

File: `register_v2.py`

In [ ]:
v2 = mlflow.genai.register_prompt(
    name="support-system-prompt",
    template="You are a support agent for {{ product }}. State the direct answer in "
             "your first sentence, then add any conditions. Refund policy, effective "
             "July 2026, is 14 days from purchase for all plans. "
             "Question: {{ question }}",
    commit_message="v2, lead with the answer; current refund window "
                   "(evidence: traces tr-7f3a9c, tr-b21e40)",
)


## Compare v1 and v2 on the dataset

File: `compare_versions.py`

Not defined here: `agent` is your agent; `dataset` comes from `regression_dataset.py`; `cites_current_policy` and `leads_with_answer` come from Part 1's `scorers.py`.

In [ ]:
from mlflow.genai.scorers import Correctness


def predict_with(version: int):
    prompt = mlflow.genai.load_prompt(f"prompts:/support-system-prompt/{version}")

    def predict(question: str) -> str:
        return agent.invoke(prompt.format(product="Acme", question=question))
    return predict


scorers = [cites_current_policy, leads_with_answer, Correctness()]

v1 = mlflow.genai.evaluate(
    data=dataset.df, predict_fn=predict_with(1), scorers=scorers
)
v2 = mlflow.genai.evaluate(
    data=dataset.df, predict_fn=predict_with(2), scorers=scorers
)

for name in ("cites_current_policy/mean", "leads_with_answer/mean"):
    print(name, v1.metrics[name], "->", v2.metrics[name])


## Run the dataset as tests

File: `test_support_agent_regressions.py`

Not defined here: `run_agent` comes from `support_agent.py`; the scorers from Part 1's `scorers.py`.

In [ ]:
import mlflow
import pytest
from mlflow.genai.datasets import get_dataset

dataset = get_dataset(name="support_regressions")


@mlflow.test
@pytest.mark.parametrize(
    "row", dataset.df.to_dict("records"),
    ids=lambda r: r["inputs"]["question"][:40],
)
def test_support_agent_regressions(row):
    result = mlflow.genai.evaluate(
        predict_fn=run_agent,
        data=[row],
        scorers=[cites_current_policy, leads_with_answer, Correctness()],
    )
    assert result.passed, result.reason


## Promote by moving the alias

File: `promote.py`

Not defined here: `v2` comes from `register_v2.py`.

In [ ]:
mlflow.genai.set_prompt_alias(
    name="support-system-prompt", alias="production", version=v2.version
)


## Export failing traces for a coding agent

File: `export_failing_traces.sh`

```bash
mlflow traces search --experiment-id 1 \
  --filter-string "feedback.quality = 'Fail'" \
  --output json > /tmp/failing_week37.json
```

## Tests a coding agent wrote for one failure theme

File: `test_refund_window.py`

Not defined here: `run_agent` comes from `support_agent.py`; `leads_with_answer` from Part 1's `scorers.py`; `RegexMatch` from `mlflow.genai.scorers`.

In [ ]:
# Generated by the coding agent for theme 3 of 5, "buries the answer"
@mlflow.test
@pytest.mark.parametrize(
    "question, must_match",
    [("How long can I wait to request a refund?", r"14 days"),
     ("Do annual plans get a longer refund window?", r"14 days")],
    ids=["refund_window", "annual_plan_window"],
)
def test_should_lead_with_refund_window(question, must_match):
    result = mlflow.genai.evaluate(
        predict_fn=run_agent,
        data=[{"inputs": {"question": question}}],
        scorers=[
            RegexMatch(name="mentions_window", pattern=must_match),
            leads_with_answer,
        ],
    )
    assert result.passed, result.reason


@mlflow.test
def test_should_not_lead_with_window_when_not_asked():
    result = mlflow.genai.evaluate(
        predict_fn=run_agent,
        data=[{"inputs": {"question": "How do I change my billing email?"}}],
        scorers=[RegexMatch(name="no_window", pattern=r"^(?!.*14 days)")],
    )
    assert result.passed, result.reason


## Run the tests in CI

File: `regression-job.yml`

```yaml
  regression:
    runs-on: ubuntu-latest
    env:
      MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_TRACKING_URI }}
      MLFLOW_GENAI_JUDGE_DEFAULT_MODEL: "openai:/gpt-5-mini"
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
    steps:
      - uses: actions/checkout@v4
      - run: pip install "mlflow>=3.14" pytest pytest-xdist
      - run: pytest tests/ -n auto
```